# Exercise: Teach an LLM to Spell with Supervised Fine-Tuning (SFT)

Large language models (LLMs) are notoriously bad at spelling. This is partly because tokenizers break words into smaller pieces, so the model learns about sub-word units rather than whole words and their spellings.

In this exercise, you'll use supervised fine-tuning (SFT) and a technique called Parameter-Efficient Fine-Tuning (PEFT) with Low-Rank Adaptation (LoRA) to teach a small LLM how to spell words. This is a classic example of teaching a model a new skill that isn't well-represented in its pre-training data.

## What you'll do in this notebook

1.  **Setup**: Import libraries and configure the environment.
2.  **Load the tokenizer and base model**: Use a small, instruction-tuned model as our starting point.
3.  **Create the dataset**: Generate a simple dataset of words and their correct spellings.
4.  **Evaluate the base model**: Test the model's spelling ability *before* fine-tuning to establish a baseline.
5.  **Configure LoRA and train**: Attach a LoRA adapter to the model and fine-tune it on the spelling dataset.
6.  **Evaluate the fine-tuned model**: Test the model again to see if its spelling has improved.

## Setup

In [71]:
# Setup imports
# No changes needed in this cell

import os
import torch
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# Use GPU, MPS, or CPU, in that order of preference
if torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon
else:
    device = torch.device("cpu")
torch.set_num_threads(max(1, os.cpu_count() // 2))
print("Using device:", device)

Using device: cuda


## Step 1. Load the tokenizer and base model

The model `HuggingFaceTB/SmolLM2-135M-Instruct` is a small, instruction-tuned model that's suitable for this exercise. It has 135 million parameters, making it lightweight and efficient for fine-tuning. It's not the most powerful model, but it's a good choice for demonstrating the concepts of SFT and PEFT with LoRA, especially on a CPU or limited GPU resources.

In [72]:
# Student task: Load the model and tokenizer, and copy the model to the device.
# TODO: Complete the sections with **********

# See: https://huggingface.co/docs/transformers/en/models
# See: https://huggingface.co/docs/transformers/en/fast_tokenizers

# Model ID for SmolLM2-135M-Instruct
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
)

# Copy the model to the device (GPU, MPS, or CPU)
model = model.to(device)



print("Model parameters (total):", sum(p.numel() for p in model.parameters()))

Model parameters (total): 134515008


## Step 2. Create the dataset

In [73]:
# Create a list of words of different lengths
# No changes are needed in this cell.

# fmt: off
ALL_WORDS = [
    "idea", "glow", "rust", "maze", "echo", "wisp", "veto", "lush", "gaze", "knit", "fume", "plow",
    "void", "oath", "grim", "crisp", "lunar", "fable", "quest", "verge", "brawn", "elude", "aisle",
    "ember", "crave", "ivory", "mirth", "knack", "wryly", "onset", "mosaic", "velvet", "sphinx",
    "radius", "summit", "banner", "cipher", "glisten", "mantle", "scarab", "expose", "fathom",
    "tavern", "fusion", "relish", "lantern", "enchant", "torrent", "capture", "orchard", "eclipse",
    "frescos", "triumph", "absolve", "gossipy", "prelude", "whistle", "resolve", "zealous",
    "mirage", "aperture", "sapphire",
]
# fmt: on

In [74]:
# Student Task: Create a Hugging Face Dataset with the prompt that asks the model to spell the word
# with hyphens between the letters.
# TODO: Complete the sections with **********


def generate_records():
    for word in ALL_WORDS:
        yield {
            "prompt": (
                f"Spell the word using hyphens between each letter.\n"
                f"Example: CAT -> C-A-T.\n"
                f"Word: {word}\n"
                f"Spelling:"
            ),
            # leading space is important for tokenization
            "completion": " " + "-".join(word).upper() + ".",
        }


ds = Dataset.from_generator(generate_records)

# Show the first item
ds[0]

Generating train split: 62 examples [00:00, 4988.43 examples/s]


{'prompt': 'Spell the word using hyphens between each letter.\nExample: CAT -> C-A-T.\nWord: idea\nSpelling:',
 'completion': ' I-D-E-A.'}

In [75]:
# Student Task: Split the dataset into training and testing sets
# See: train_test_split
# TODO: Complete the sections with **********

ds = ds.train_test_split(test_size=0.25, seed=42) # Set the test set to be 25% of the dataset, and the rest is training



In [76]:
# View the training set
# No changes needed in this cell

ds["train"]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 46
})

## Step 3. Evaluate the base model

Before we fine-tune the model, let's see how it performs on the spelling task. We'll create a helper function to generate a spelling for a given word and compare it to the correct answer.

In [77]:
# Student task: Create a function to check the model's spelling.
# This function will take a model, tokenizer, prompt, and the correct spelling.
# It should generate text from the model and compare the model's proposed spelling
# to the actual spelling, returning the proportion of characters that were correct.
# TODO: Complete the sections with **********


def check_spelling(
    model, tokenizer, prompt: str, actual_spelling: str, max_new_tokens: int = 30
) -> float:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=False,
    )

    # Only decode newly generated tokens
    generated_tokens = gen[0][inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Take first line only
    proposed_spelling = output.strip().split("\n")[0].strip()

    expected = actual_spelling.strip()

    match = proposed_spelling == expected

    print(
        f"Proposed: {proposed_spelling} | Actual: {expected} "
        f"| Matches: {'✅' if match else '❌'}"
    )

    # Return 1.0 or 0.0 instead of partial credit
    return 1.0 if match else 0.0


check_spelling(
    model=model,
    tokenizer=tokenizer,
    prompt=ds["test"][0]["prompt"],
    actual_spelling=ds["test"][0]["completion"],
)

Proposed: wryly | Actual: W-R-Y-L-Y. | Matches: ❌


0.0

In [78]:
# Student task: Evaluate the base model's spelling ability
# We expect it to perform poorly, as it hasn't been trained for this task.

proportion_correct = 0.0

for example in ds["train"].select(range(20)):
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=10,
    )
    proportion_correct += result

print(f"{proportion_correct}/20.0 words correct")

Proposed: sphinx | Actual: S-P-H-I-N-X. | Matches: ❌
Proposed: brawn | Actual: B-R-A-W-N. | Matches: ❌
Proposed: gossipy | Actual: G-O-S-S-I-P-Y. | Matches: ❌
Proposed: enchant | Actual: E-N-C-H-A-N-T. | Matches: ❌
Proposed: tavern | Actual: T-A-V-E-R-N. | Matches: ❌
Proposed: whistle | Actual: W-H-I-S-T-L-E. | Matches: ❌
Proposed: capture | Actual: C-A-P-T-U-R-E. | Matches: ❌
Proposed: echo | Actual: E-C-H-O. | Matches: ❌
Proposed: mir-th | Actual: M-I-R-T-H. | Matches: ❌
Proposed: crisp | Actual: C-R-I-S-P. | Matches: ❌
Proposed: z-e-l-l-o- | Actual: Z-E-A-L-O-U-S. | Matches: ❌
Proposed: em-ber | Actual: E-M-B-E-R. | Matches: ❌
Proposed: scarab | Actual: S-C-A-R-A-B. | Matches: ❌
Proposed: knit | Actual: K-N-I-T. | Matches: ❌
Proposed: resolve | Actual: R-E-S-O-L-V-E. | Matches: ❌
Proposed: VEL-U-T | Actual: V-E-L-V-E-T. | Matches: ❌
Proposed: absolve | Actual: A-B-S-O-L-V-E. | Matches: ❌
Proposed: lunar | Actual: L-U-N-A-R. | Matches: ❌
Proposed: Maze | Actual: M-A-Z-E. | Matches: ❌

As expected, the base model is terrible at spelling. It mostly just repeats the word back. Now, let's fine-tune it.

## Step 4. Configure LoRA and train the model

Let’s attach a LoRA adapter to the base model. We use a LoRA config so only a tiny fraction of parameters are trainable. Read more here: [LoRA](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora).

In [79]:
# Student task: Configure LoRA for a causal LM and wrap the model with get_peft_model
# Complete the sections with **********

# Print how many params are trainable at first
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params BEFORE: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)
from peft import LoraConfig, get_peft_model, TaskType
# See: https://huggingface.co/docs/peft/package_reference/lora
# lora_config = LoraConfig(
#     r=**********,                 # Rank of the update matrices. Lower value = fewer trainable parameters.
#     lora_alpha=**********,        # LoRA scaling factor.
#     lora_dropout=**********,      # Dropout probability for LoRA layers.
#     bias="none",
#     task_type=**********,         # Causal Language Modeling.
# )
# # Wrap the base model with get_peft_model
# model = get_peft_model(**********, **********)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Print the number of trainable parameters after applying LoRA
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params AFTER: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)

Trainable params BEFORE: 134,515,008 / 134,515,008 (100.00%)
Trainable params AFTER: 460,800 / 134,975,808 (0.34%)


Now let’s set the training arguments. We'll use `SFTConfig` from the TRL library, which is a wrapper around the standard `TrainingArguments`. We keep epochs, batch size, and sequence length modest to finish training quickly.

In [80]:
# Student task: Fill in the SFTConfig for a quick training run
# Complete the sections with **********

output_dir = "data/model"

# See: https://huggingface.co/docs/trl/en/sft_trainer#trl.SFTConfig
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=20,          # 🔥 increased
    learning_rate=1e-4,           # 🔥 more stable
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    report_to=[],
    fp16=False,
    lr_scheduler_type="cosine",
)


Now we define the `SFTTrainer` and run the fine-tuning process.

In [81]:
# Student Task: Create and run the SFTTrainer
# TODO: Complete the sections with **********


# See: https://huggingface.co/docs/trl/en/sft_trainer
trainer = SFTTrainer(
     model=model,
     train_dataset=ds["train"],
     eval_dataset=ds["test"],
     args=training_args,
 )
# Now train it:
trainer.train()


Truncating eval dataset: 100%|██████████| 16/16 [00:00<00:00, 547.23 examples/s]


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,1.445100,1.336982,2.712993,1696.000000,0.560638
20,1.349100,1.213609,2.695724,3315.000000,0.627969
30,1.147500,1.106642,2.648298,4948.000000,0.680027
40,1.119900,1.018862,2.603753,6537.000000,0.699351
50,1.004400,0.944190,2.579919,8146.000000,0.716234
60,0.927400,0.878788,2.561595,9765.000000,0.736941
70,0.880800,0.830158,2.528377,11460.000000,0.757070
80,0.857200,0.789164,2.503865,13076.000000,0.777551
90,0.740900,0.762317,2.466291,14693.000000,0.771538
100,0.810500,0.742420,2.445671,16318.000000,0.766908


TrainOutput(global_step=240, training_loss=0.8305985987186432, metrics={'train_runtime': 65.5761, 'train_samples_per_second': 14.03, 'train_steps_per_second': 3.66, 'total_flos': 26501801045760.0, 'train_loss': 0.8305985987186432, 'epoch': 20.0})

## Step 5. Evaluate the fine-tuned model

In [82]:
# Evaluate the fine-tuned model on the same training examples
# No changes needed in this cell


proportion_correct = 0.0

for example in ds["train"].select(range(20)):
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=30,
    )
    proportion_correct += result

print(f"{proportion_correct}/20.0 words correct")

Proposed: S-P-H-N-I-X. | Actual: S-P-H-I-N-X. | Matches: ❌
Proposed: B-R-A-W-N. | Actual: B-R-A-W-N. | Matches: ✅
Proposed: G-S-E-N-O-P-I-Y. | Actual: G-O-S-S-I-P-Y. | Matches: ❌
Proposed: E-N-C-H-T-E-N. | Actual: E-N-C-H-A-N-T. | Matches: ❌
Proposed: A-T-R-A-T. | Actual: T-A-V-E-R-N. | Matches: ❌
Proposed: W-H-I-S-T. | Actual: W-H-I-S-T-L-E. | Matches: ❌
Proposed: C-A-P-U-S. | Actual: C-A-P-T-U-R-E. | Matches: ❌
Proposed: E-R-E-H-O-T. | Actual: E-C-H-O. | Matches: ❌
Proposed: M-R-I-T-H. | Actual: M-I-R-T-H. | Matches: ❌
Proposed: C-I-S-T. | Actual: C-R-I-S-P. | Matches: ❌
Proposed: Z-E-O-L-E. | Actual: Z-E-A-L-O-U-S. | Matches: ❌
Proposed: E-M-B-E. | Actual: E-M-B-E-R. | Matches: ❌
Proposed: S-A-R-K-A-B. | Actual: S-C-A-R-A-B. | Matches: ❌
Proposed: K-N-I-T. | Actual: K-N-I-T. | Matches: ✅
Proposed: R-E-S-S-I-L-E. | Actual: R-E-S-O-L-V-E. | Matches: ❌
Proposed: V-E-U-T. | Actual: V-E-L-V-E-T. | Matches: ❌
Proposed: A-B-O-R-E-V-E. | Actual: A-B-S-O-L-V-E. | Matches: ❌
Proposed: L-A-M-O

The model now performs better on the training data it has seen. But has it generalized? Let's check its performance on the unseen test set.

In [83]:
# Evaluate the fine-tuned model on the unseen test set
# No changes needed in this cell


proportion_correct = 0.0
num_examples = len(ds["test"])

for example in ds["test"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    proportion_correct += result

print(f"{proportion_correct}/{num_examples}.0 words correct")

Proposed: W-R-Y-I-L-Y. | Actual: W-R-Y-L-Y. | Matches: ❌
Proposed: G-L-I-S-H-E-N. | Actual: G-L-I-S-T-E-N. | Matches: ❌
Proposed: Q-A-S-E-R-T-E. | Actual: Q-U-E-S-T. | Matches: ❌
Proposed: C-A-R-E. | Actual: C-R-A-V-E. | Matches: ❌
Proposed: L-I-S-T. | Actual: L-U-S-H. | Matches: ❌
Proposed: F-A-L-I-C-E. | Actual: F-A-B-L-E. | Matches: ❌
Proposed: K-A-N-K-R-E. | Actual: K-N-A-C-K. | Matches: ❌
Proposed: T-O-V-E-R. | Actual: T-R-I-U-M-P-H. | Matches: ❌
Proposed: S-P-O-F-I-R-E. | Actual: S-A-P-P-H-I-R-E. | Matches: ❌
Proposed: E-S-P-E-N-O-S. | Actual: E-X-P-O-S-E. | Matches: ❌
Proposed: F-R-O-S-C-O-N. | Actual: F-R-E-S-C-O-S. | Matches: ❌
Proposed: W-I-S-P. | Actual: W-I-S-P. | Matches: ✅
Proposed: M-I-E-R-A-G. | Actual: M-I-R-A-G-E. | Matches: ❌
Proposed: I-V-O-L-E. | Actual: I-V-O-R-Y. | Matches: ❌
Proposed: O-N-D-I-S-T. | Actual: O-N-S-E-T. | Matches: ❌
Proposed: E-L-A-U-D. | Actual: E-L-U-D-E. | Matches: ❌
1.0/16.0 words correct


It looks like it has improved! Perhaps with a larger dataset and more training, it could get even better.

## Congratulations for completing the exercise! 🎉

✅ You did it! You successfully fine-tuned a small language model using PEFT with LoRA to teach it a new skill: spelling! You saw how the base model failed completely at the task, and with a very small amount of data and a short training run, the model managed to get better at spelling.

<br /><br /><br /><br /><br /><br /><br /><br /><br />